# Research 34 — Four-seed VIP × region OOF average

**Hypothesis:** a fourth CV partition may further stabilize the VIP × region OOF ensemble. Compare fixed 1/3 and 1/4 equal-weight averages, both contributing 20% to the current reference. Component OOF predictions are aligned row-wise; this uses train labels only and is not independent validation. No model fitting or weight search.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    values = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(values).all() and (values >= 0).all(), filename
    return values / values.sum(axis=1, keepdims=True)


p_base = load_probs("segment-oof.csv")
p42 = load_probs("segment-vip-region-balanced-oof.csv")
p2025 = load_probs("segment-vip-region-seed-2025-oof.csv")
p1337 = load_probs("segment-vip-region-seed-1337-oof.csv")
p31415 = load_probs("segment-vip-region-seed-31415-oof.csv")
p_three = (p42 + p2025 + p1337) / 3
p_four = (p42 + p2025 + p1337 + p31415) / 4
candidates = {
    "Current reference": p_base,
    "80/20 three-seed average": 0.8 * p_base + 0.2 * p_three,
    "80/20 four-seed average": 0.8 * p_base + 0.2 * p_four,
}
fold_sets = {
    "seed42": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y)
    ),
    "seed2025": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=2025).split(np.zeros(len(y)), y)
    ),
    "seed1337": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=1337).split(np.zeros(len(y)), y)
    ),
    "seed31415": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=31415).split(np.zeros(len(y)), y)
    ),
}
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for seed_name, folds in fold_sets.items():
        for fold, (_, idx) in enumerate(folds, start=1):
            row[f"{seed_name}_fold_{fold}_macro_f1"] = f1_score(y[idx], pred[idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-vip-region-four-seed-average.csv", index=False)
pd.DataFrame(candidates["80/20 four-seed average"], columns=[f"prob_{i}" for i in range(4)]).assign(
    segment=y
).to_csv(ROOT / "predictions/segment-vip-region-four-seed-average-oof.csv", index=False)
print("Four-seed OOF synthesis complete; Test not loaded.")

,candidate,macro_f1,weighted_f1,seed42_fold_1_macro_f1,seed42_fold_2_macro_f1,seed42_fold_3_macro_f1,seed2025_fold_1_macro_f1,seed2025_fold_2_macro_f1,seed2025_fold_3_macro_f1,seed1337_fold_1_macro_f1,seed1337_fold_2_macro_f1,seed1337_fold_3_macro_f1,seed31415_fold_1_macro_f1,seed31415_fold_2_macro_f1,seed31415_fold_3_macro_f1
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.750797,0.749412,0.752462,0.749956,0.753540,0.749163,0.749806,0.750854,0.752005
1,80/20 three-seed average,0.751862,0.765462,0.750224,0.750851,0.754482,0.751465,0.750765,0.753346,0.751549,0.754312,0.749708,0.750866,0.751830,0.752878
2,80/20 four-seed average,0.752172,0.765704,0.750552,0.751365,0.754568,0.751751,0.751072,0.753684,0.752052,0.754397,0.750051,0.751079,0.752230,0.753192


Four-seed OOF synthesis complete; Test not loaded.
